In [9]:
import torch
from datasets import load_dataset
ds = load_dataset('json', data_files="/home/sam/torch/data/alpaca.jsonl")

In [10]:
ds = ds['train'].train_test_split(test_size=0.2)

In [11]:
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 16543
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 4136
    })
})

In [12]:
from torch.utils.data import Dataset
import pandas as pd
ds = pd.read_json("data/alpaca.jsonl", lines=True)
def format_example(row):
    return f"""
### Instruction:
{row['instruction']}

### Input:
{row['input']}

### Response:
{row['output']}
"""
class AlpacaDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df.apply(format_example, axis=1).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": enc["input_ids"].squeeze()
        }

In [13]:
ds = AlpacaDataset(ds, tokenizer)

In [14]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model
)

import torch

In [15]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [24]:

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct",
                                             device_map='auto',
                                             quantization_config=bnb_config)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [17]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [18]:
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [19]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit"
)

In [20]:
trainer = Trainer(
    model,
    args,
    train_dataset=ds
)

In [21]:
#trainer.train()

In [30]:
from peft import PeftModel

In [33]:
model = PeftModel.from_pretrained(model, "./results/checkpoint-18900")

In [34]:
prompt = "### Instruction:\nExplain gravity simply.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Explain gravity simply.

### Response:
gravity is the force that attracts objects towards each other. it is caused by the mass of an object and is strongest when the masses are close together. gravity causes objects to fall to the ground when they are dropped, and it also causes planets to orbit around the sun. it is a fundamental force in the universe and plays a crucial role in shaping the structure of galaxies and the motion of celestial bodies.
In simpler terms, gravity is the invisible force that pulls objects towards each other. It's what makes
